# Date exploration

In [5]:
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from datetime import datetime

from utils import extract_date_from_recording_id

In [6]:
data_path = "data/dataset_with_aug_dict.parquet"
df = pd.read_parquet(data_path)


In [7]:
df.head()

,recording_id,start_time,end_time,augmentation_dict,audspec_lengthL1norm_sma_iqr1_2,audspec_lengthL1norm_sma_iqr2_3,audspec_lengthL1norm_sma_iqr1_3,audspec_lengthL1norm_sma_percentile50_0,audspec_lengthL1norm_sma_stddev,audspec_lengthL1norm_sma_amean,...,logHNR_sma_de_iqr2_3,logHNR_sma_de_iqr1_3,logHNR_sma_de_percentile50_0,logHNR_sma_de_stddev,F0final_sma_ff0_nnz,patient_short_id,label,age,sex,audio_quality
0,patient_0002/2021-10-01,480,680,"{""augmentation"": ""original""}",0.824093,0.237206,1.061300,1.294769,0.601383,1.036854,...,0.178952,0.204519,0.065431,0.157982,1.000000,patient_0002,0,0.909091,0,0.557534
1,patient_0002/2021-10-01,480,680,"{""pitch_shift"": null, ""time_stretch"": null, ""r...",0.825607,0.208997,1.034604,1.211626,0.564895,0.933170,...,0.133620,0.190211,0.089150,0.180251,1.000000,patient_0002,0,0.909091,0,0.557534
2,patient_0002/2021-10-01,480,680,"{""pitch_shift"": 0.31846850944034655, ""time_str...",0.655419,0.376667,1.032087,1.198619,0.577098,1.036244,...,5.281692,5.338002,0.086555,3.712759,0.846154,patient_0002,0,0.909091,0,0.557534
3,patient_0002/2021-10-01,480,680,"{""pitch_shift"": null, ""time_stretch"": null, ""r...",0.679861,0.330526,1.010386,1.171579,0.563352,0.997027,...,0.221065,0.251099,0.082160,0.181627,1.000000,patient_0002,0,0.909091,0,0.557534
4,patient_0002/2021-10-01,576,776,"{""pitch_shift"": null, ""time_stretch"": 0.835476...",0.635661,0.266532,0.902193,1.221095,0.526917,1.019168,...,5.344244,5.419852,0.062196,3.722184,0.846154,patient_0002,0,0.909091,0,0.809958


In [8]:
print(f"Shape of the dataset: {df.shape}")
unique_patients = df["patient_short_id"].unique()
print(f"Number of unique patients: {len(unique_patients)}")
print(f"Number of unique recording dates: {df['recording_id'].apply(extract_date_from_recording_id).nunique()}")
print("Columns in the dataset:")
for col in df.columns:
    print(f" - {col}")

Shape of the dataset: (73542, 790)
Number of unique patients: 14
Number of unique recording dates: 248
Columns in the dataset:
 - recording_id
 - start_time
 - end_time
 - augmentation_dict
 - audspec_lengthL1norm_sma_iqr1_2
 - audspec_lengthL1norm_sma_iqr2_3
 - audspec_lengthL1norm_sma_iqr1_3
 - audspec_lengthL1norm_sma_percentile50_0
 - audspec_lengthL1norm_sma_stddev
 - audspec_lengthL1norm_sma_amean
 - audspecRasta_lengthL1norm_sma_iqr1_2
 - audspecRasta_lengthL1norm_sma_iqr2_3
 - audspecRasta_lengthL1norm_sma_iqr1_3
 - audspecRasta_lengthL1norm_sma_percentile50_0
 - audspecRasta_lengthL1norm_sma_stddev
 - audspecRasta_lengthL1norm_sma_amean
 - pcm_RMSenergy_sma_iqr1_2
 - pcm_RMSenergy_sma_iqr2_3
 - pcm_RMSenergy_sma_iqr1_3
 - pcm_RMSenergy_sma_percentile50_0
 - pcm_RMSenergy_sma_stddev
 - pcm_RMSenergy_sma_amean
 - pcm_zcr_sma_iqr1_2
 - pcm_zcr_sma_iqr2_3
 - pcm_zcr_sma_iqr1_3
 - pcm_zcr_sma_percentile50_0
 - pcm_zcr_sma_stddev
 - pcm_zcr_sma_amean
 - audspec_lengthL1norm_sma_de_i

In [9]:
# Recordings per patient
recordings_per_patient = df.groupby("patient_short_id").size()
print("\nRecordings per patient:")
print(f"  Mean: {recordings_per_patient.mean():.1f}")
print(f"  Median: {recordings_per_patient.median():.1f}")
print(f"  Min: {recordings_per_patient.min()}")
print(f"  Max: {recordings_per_patient.max()}")


Recordings per patient:
  Mean: 5253.0
  Median: 4765.5
  Min: 1407
  Max: 13946


In [10]:
# Label distribution
print(f"\n{'=' * 70}")
print("LABEL DISTRIBUTION")
print("=" * 70)
label_counts = df["label"].value_counts()
print("Overall:")
print(f"  Stable (0): {label_counts[0]:,} ({label_counts[0] / len(df) * 100:.1f}%)")
print(
    f"  Pre-hospitalization (1): {label_counts[1]:,} ({label_counts[1] / len(df) * 100:.1f}%)"
)

print("\nPer patient:")
for patient in sorted(unique_patients):
    patient_df = df[df["patient_short_id"] == patient]
    patient_labels = patient_df["label"].value_counts()
    n_stable = patient_labels.get(0, 0)
    n_prehosp = patient_labels.get(1, 0)
    print(
        f"  {patient}: {n_stable} stable, {n_prehosp} pre-hosp "
        f"({n_prehosp / (n_stable + n_prehosp) * 100:.1f}% positive)"
    )


LABEL DISTRIBUTION
Overall:
  Stable (0): 62,614 (85.1%)
  Pre-hospitalization (1): 10,928 (14.9%)

Per patient:
  patient_0000: 6654 stable, 722 pre-hosp (9.8% positive)
  patient_0001: 3985 stable, 818 pre-hosp (17.0% positive)
  patient_0002: 13332 stable, 614 pre-hosp (4.4% positive)
  patient_0003: 2729 stable, 910 pre-hosp (25.0% positive)
  patient_0004: 5223 stable, 397 pre-hosp (7.1% positive)
  patient_0005: 5525 stable, 675 pre-hosp (10.9% positive)
  patient_0006: 2004 stable, 845 pre-hosp (29.7% positive)
  patient_0007: 1135 stable, 272 pre-hosp (19.3% positive)
  patient_0008: 1319 stable, 393 pre-hosp (23.0% positive)
  patient_0009: 2263 stable, 643 pre-hosp (22.1% positive)
  patient_0010: 3763 stable, 965 pre-hosp (20.4% positive)
  patient_0011: 1393 stable, 1309 pre-hosp (48.4% positive)
  patient_0012: 4993 stable, 1464 pre-hosp (22.7% positive)
  patient_0013: 8296 stable, 901 pre-hosp (9.8% positive)


In [11]:
# Temporal information
df["recording_date"] = extract_date_from_recording_id(df["recording_id"])

print("\nRecording date range:")
print(f"  First recording: {df['recording_date'].min().date()}")
print(f"  Last recording: {df['recording_date'].max().date()}")
print(
    f"  Total span: {(df['recording_date'].max() - df['recording_date'].min()).days} days"
)

print("\nFollow-up period per patient:")
for patient in sorted(unique_patients):
    patient_df = df[df["patient_short_id"] == patient]
    first_date = patient_df["recording_date"].min()
    last_date = patient_df["recording_date"].max()
    follow_up_days = (last_date - first_date).days
    n_recordings = len(patient_df)
    print(
        f"  {patient}: {follow_up_days} days ({first_date.date()} to {last_date.date()}), {n_recordings} recordings"
    )


Recording date range:
  First recording: 2021-09-13
  Last recording: 2022-08-21
  Total span: 342 days

Follow-up period per patient:
  patient_0000: 327 days (2021-09-17 to 2022-08-10), 7376 recordings
  patient_0001: 196 days (2021-09-21 to 2022-04-05), 4803 recordings
  patient_0002: 320 days (2021-10-01 to 2022-08-17), 13946 recordings
  patient_0003: 296 days (2021-09-17 to 2022-07-10), 3639 recordings
  patient_0004: 342 days (2021-09-13 to 2022-08-21), 5620 recordings
  patient_0005: 327 days (2021-09-26 to 2022-08-19), 6200 recordings
  patient_0006: 90 days (2021-12-17 to 2022-03-17), 2849 recordings
  patient_0007: 209 days (2021-10-30 to 2022-05-27), 1407 recordings
  patient_0008: 97 days (2021-12-29 to 2022-04-05), 1712 recordings
  patient_0009: 186 days (2022-02-07 to 2022-08-12), 2906 recordings
  patient_0010: 168 days (2022-01-29 to 2022-07-16), 4728 recordings
  patient_0011: 247 days (2021-11-04 to 2022-07-09), 2702 recordings
  patient_0012: 278 days (2021-11-10 

In [12]:
# Feature information
print(f"\n{'=' * 70}")
print("FEATURE INFORMATION")
print("=" * 70)

if "augmentation_dict" in df.columns :
    metadata_cols = [
    "recording_id",
    "patient_short_id",
    "label",
    "recording_date",  # Added by our extraction
    "augmentation_dict"
    ]
else :
    metadata_cols = [
        "recording_id",
        "patient_short_id",
        "label",
        "recording_date",  # Added by our extraction
    ]

feature_cols = [col for col in df.columns if col not in metadata_cols]

print(f"Number of acoustic features: {len(feature_cols)}")
print("\nFeature categories:")


FEATURE INFORMATION
Number of acoustic features: 786

Feature categories:


In [13]:
# Count feature types
feature_types = {}
for col in feature_cols:
    feature_type = col.split("_")[0]
    feature_types[feature_type] = feature_types.get(feature_type, 0) + 1

for ftype, count in sorted(feature_types.items()):
    print(f"  {ftype}: {count} features")

  F0final: 13 features
  age: 1 features
  audSpec: 312 features
  audio: 1 features
  audspec: 12 features
  audspecRasta: 12 features
  end: 1 features
  jitterDDP: 12 features
  jitterLocal: 12 features
  logHNR: 12 features
  mfcc: 168 features
  pcm: 204 features
  sex: 1 features
  shimmerLocal: 12 features
  start: 1 features
  voicingFinalUnclipped: 12 features


## Select top features

In [14]:
# Number of top features to select
X = df[feature_cols]  # Features dataframe
y = df["label"]  # Target labels

num_features = 300
print(f"\nSelecting top {num_features} features among {X.shape[-1]} based on ANOVA F-value")

# Feature selection using ANOVA F-value
score_func = f_classif

selector = SelectKBest(score_func=score_func, k=num_features)
X_selected = selector.fit_transform(X, y)
selected_feature_names = X.columns[selector.get_support()]

# Dataframe with selected top features
df_selected_features = pd.DataFrame(X_selected, columns=selected_feature_names)


Selecting top 300 features among 786 based on ANOVA F-value


In [15]:
df_selected_features.head()

,start_time,end_time,audspec_lengthL1norm_sma_iqr1_2,audspec_lengthL1norm_sma_iqr2_3,audspec_lengthL1norm_sma_iqr1_3,audspec_lengthL1norm_sma_percentile50_0,audspec_lengthL1norm_sma_stddev,audspec_lengthL1norm_sma_amean,pcm_RMSenergy_sma_iqr1_2,pcm_RMSenergy_sma_iqr1_3,...,shimmerLocal_sma_de_iqr1_3,shimmerLocal_sma_de_percentile50_0,shimmerLocal_sma_de_stddev,logHNR_sma_de_amean,logHNR_sma_de_iqr1_2,logHNR_sma_de_iqr2_3,logHNR_sma_de_iqr1_3,logHNR_sma_de_percentile50_0,F0final_sma_ff0_nnz,age
0,480.0,680.0,0.824093,0.237206,1.061300,1.294769,0.601383,1.036854,0.128371,0.200574,...,0.013862,-0.005926,0.007873,0.138261,0.025567,0.178952,0.204519,0.065431,1.000000,0.909091
1,480.0,680.0,0.825607,0.208997,1.034604,1.211626,0.564895,0.933170,0.169030,0.224128,...,0.010074,-0.009416,0.005585,0.090609,0.056591,0.133620,0.190211,0.089150,1.000000,0.909091
2,480.0,680.0,0.655419,0.376667,1.032087,1.198619,0.577098,1.036244,0.065685,0.184966,...,0.006873,-0.004433,0.005731,2.765766,0.056310,5.281692,5.338002,0.086555,0.846154,0.909091
3,480.0,680.0,0.679861,0.330526,1.010386,1.171579,0.563352,0.997027,0.096811,0.182359,...,0.005170,-0.004470,0.005559,0.177026,0.030034,0.221065,0.251099,0.082160,1.000000,0.909091
4,576.0,776.0,0.635661,0.266532,0.902193,1.221095,0.526917,1.019168,0.038293,0.093220,...,0.008177,-0.002504,0.007473,2.755419,0.075607,5.344244,5.419852,0.062196,0.846154,0.909091


In [16]:
# Reconstruct the final dataframe with metadata and selected features
df_final = pd.concat([df[metadata_cols].reset_index(drop=True), df_selected_features], axis=1)
print(f"\nFinal dataframe shape with selected features: {df_final.shape}")

# Save the final dataframe to a new parquet file
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
final_data_path = f"data/dataset_{num_features}_selected_features_{current_time}.parquet"
df_final.to_parquet(final_data_path)
print(f"Final dataframe saved to {final_data_path}")


Final dataframe shape with selected features: (73542, 305)
Final dataframe saved to data/dataset_300_selected_features_20260122_105307.parquet
